# Robot Pose + Health Monitoring Demo
RoboKeyGen-inspired keypoint lifting + RoboPose-inspired render-and-compare, followed by multimodal fault diagnosis. Enable a GPU runtime before running.

In [ ]:
!pip -q install numpy scipy scikit-learn joblib Pillow opencv-python-headless matplotlib torch torchvision einops imageio imageio-ffmpeg
import os, torch
os.environ['PYTHONPATH'] = os.path.abspath('src')
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!PYTHONPATH=src python -m robot_demo.simulator --output artifacts/pose_dataset.npz --samples 4000

In [ ]:
!PYTHONPATH=src python -m robot_demo.train_pose --data artifacts/pose_dataset.npz --epochs 20 --batch-size 128

In [ ]:
!PYTHONPATH=src python -m robot_demo.train_fault --output artifacts/fault_model.joblib --samples 8000
!PYTHONPATH=src python -m robot_demo.run_demo --fault gearbox_backlash --joint 3

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/pose_example.png'))
display(Image('artifacts/demo_result.png'))
print(open('artifacts/demo_summary.json').read())

## Optional: ABB temporal-pose video
Upload the personal-use ABB source clip as `artifacts/abb_source_personal_use.mp4`. This runs CoTracker3; it is not a calibrated ABB RoboPose model.

In [ ]:
!git clone -q https://github.com/facebookresearch/co-tracker.git ../cotracker_upstream
!git -C ../cotracker_upstream checkout -q 82e02e8029753ad4ef13cf06be7f4fc5facdda4d
!mkdir -p ../cotracker_upstream/checkpoints
!wget -q -O ../cotracker_upstream/checkpoints/scaled_offline.pth https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth
!PYTHONPATH=src python -m robot_demo.make_real_robot_video_v2 --input-video artifacts/abb_source_personal_use.mp4 --artifacts artifacts --output artifacts/abb_real_robot_demo_source_v2.avi --cotracker-repo ../cotracker_upstream --checkpoint ../cotracker_upstream/checkpoints/scaled_offline.pth --tracking-stride 2 --force-track
!ffmpeg -y -loglevel error -i artifacts/abb_real_robot_demo_source_v2.avi -c:v libx264 -crf 18 -pix_fmt yuv420p -movflags +faststart -an artifacts/abb_real_robot_demo.mp4
from IPython.display import Video
Video('artifacts/abb_real_robot_demo.mp4', embed=True)